In [6]:
import torch
import torch.nn.functional as F

# 1. 定义原始数据：模拟一个 5x5 的输入图像
input = torch.tensor([[1, 2, 0, 3, 1], 
                      [0, 1, 2, 3, 1], 
                      [1, 2, 1, 0, 0], 
                      [5, 2, 3, 1, 1], 
                      [2, 1, 0, 1, 1]])

# 2. 定义卷积核（过滤器）：3x3 的权重矩阵
kernel = torch.tensor([[1, 2, 1], 
                       [0, 1, 0], 
                       [2, 1, 0]])

# 3. 改变形状以满足 PyTorch 格式要求：(Batch, Channel, Height, Width)
# 也就是 (几张图, 几通道, 高, 宽)
input = torch.reshape(input, (1, 1, 5, 5))
kernel = torch.reshape(kernel, (1, 1, 3, 3))

print(input.shape)
print(kernel.shape)

# 4. 执行卷积操作
# input: 输入图像, kernel: 权重, stride=1: 每次滑窗移动 1 格
output = F.conv2d(input, kernel, stride = 1)

print(output)

torch.Size([1, 1, 5, 5])
torch.Size([1, 1, 3, 3])
tensor([[[[10, 12, 12],
          [18, 16, 16],
          [13,  9,  3]]]])


In [ ]:
import torch
import torchvision
from torch import nn
from torch.nn import Conv2d
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter

# 0. 准备数据 (补全部分，为了让 dataloader 有内容)
dataset = torchvision.datasets.CIFAR10(root="/workspace/data", train=False, transform=torchvision.transforms.ToTensor(), download=True)
dataloader = DataLoader(dataset, batch_size=64)

# 1. 定义网络模型
class Tudui(nn.Module):
    def __init__(self):
        super(Tudui, self).__init__()
        # 定义卷积层：输入3通道(RGB)，输出6通道，卷积核3x3
        self.conv1 = Conv2d(in_channels=3, out_channels=6, kernel_size=3, stride=1, padding=0)

    def forward(self, x):
        x = self.conv1(x)
        return x

# 2. 初始化模型和 TensorBoard
tudui = Tudui()
writer = SummaryWriter("logs")
step = 0

# 3. 循环处理数据
for data in dataloader:
    imgs, targets = data
    output = tudui(imgs)
    
    print(imgs.shape)   # 输入尺寸: torch.Size([64, 3, 32, 32])
    print(output.shape) # 输出尺寸: torch.Size([64, 6, 30, 30]) -> 尺寸变小是因为卷积没有padding

    # 记录输入图片 (3通道可以直接显示)
    writer.add_images("input", imgs, step)
    
    # 关键步骤：reshape
    # 原因：TensorBoard 只能显示 3通道(彩色) 或 1通道(黑白) 的图片。
    # 这里的 output 有 6 个通道，直接传会报错。
    # 所以将 (-1, 6, 30, 30) 重塑为 (-1, 3, 30, 30)。
    # Batch = -1 参数 -1 是自动计算
    # 结果是 Batch Size 会翻倍 64 -> 128 (把多出来的通道当成新的图片张数处理)。
    output = torch.reshape(output, (-1, 3, 30, 30))
    
    # 记录输出图片
    writer.add_images("output", output, step)
    step = step + 1

# 关闭 writer
writer.close()

torch.Size([64, 3, 32, 32])
torch.Size([64, 6, 30, 30])
torch.Size([64, 3, 32, 32])
torch.Size([64, 6, 30, 30])
torch.Size([64, 3, 32, 32])
torch.Size([64, 6, 30, 30])
torch.Size([64, 3, 32, 32])
torch.Size([64, 6, 30, 30])
torch.Size([64, 3, 32, 32])
torch.Size([64, 6, 30, 30])
torch.Size([64, 3, 32, 32])
torch.Size([64, 6, 30, 30])
torch.Size([64, 3, 32, 32])
torch.Size([64, 6, 30, 30])
torch.Size([64, 3, 32, 32])
torch.Size([64, 6, 30, 30])
torch.Size([64, 3, 32, 32])
torch.Size([64, 6, 30, 30])
torch.Size([64, 3, 32, 32])
torch.Size([64, 6, 30, 30])
torch.Size([64, 3, 32, 32])
torch.Size([64, 6, 30, 30])
torch.Size([64, 3, 32, 32])
torch.Size([64, 6, 30, 30])
torch.Size([64, 3, 32, 32])
torch.Size([64, 6, 30, 30])
torch.Size([64, 3, 32, 32])
torch.Size([64, 6, 30, 30])
torch.Size([64, 3, 32, 32])
torch.Size([64, 6, 30, 30])
torch.Size([64, 3, 32, 32])
torch.Size([64, 6, 30, 30])
torch.Size([64, 3, 32, 32])
torch.Size([64, 6, 30, 30])
torch.Size([64, 3, 32, 32])
torch.Size([64, 6, 3

# 卷积操作（Convolution）

## 1. 基本概念
卷积操作是深度学习中最重要的操作之一，特别是在计算机视觉任务中。

**核心过程**：
- 卷积核（Kernel/Filter）与输入图像的局部区域进行对齐
- 对齐后进行**按位相乘**（element-wise multiplication）
- 将相乘的结果**求和**，得到一个输出值
- 卷积核在图像上**滑动**，重复上述过程，生成输出特征图

## 2. 关键参数

### 2.1 卷积核大小（Kernel Size）
- 常见：3×3, 5×5, 7×7
- 决定了每次卷积操作的感受野大小

### 2.2 步长（Stride）
- 卷积核每次滑动的距离
- stride=1：每次移动1个像素
- stride=2：每次移动2个像素（输出尺寸减半）

### 2.3 填充（Padding）
- 在输入边缘填充0值
- padding=0：不填充（valid padding）
- padding=1：填充1圈0（same padding，保持尺寸）

### 2.4 输入/输出通道数
- **in_channels**：输入特征图的通道数（如RGB图像为3）
- **out_channels**：输出特征图的通道数（卷积核的个数）

## 3. 输出尺寸计算公式

$$
\text{Output Size} = \frac{W - K + 2P}{S} + 1
$$

其中：
- W = 输入宽度（或高度）
- K = 卷积核大小
- P = 填充大小
- S = 步长

## 4. PyTorch 实现

```python
import torch
import torch.nn as nn

# 创建2D卷积层
conv = nn.Conv2d(
    in_channels=3,      # 输入通道数（如RGB）
    out_channels=64,    # 输出通道数（卷积核数量）
    kernel_size=3,      # 卷积核大小 3×3
    stride=1,           # 步长
    padding=1           # 填充
)

# 输入张量: [batch_size, channels, height, width]
x = torch.randn(1, 3, 224, 224)  # 1张 224×224 的RGB图像
output = conv(x)                  # 输出: [1, 64, 224, 224]
```

## 5. 常见应用
- **特征提取**：提取图像的边缘、纹理等特征
- **降维**：使用stride>1降低特征图尺寸
- **增加非线性**：配合激活函数增强模型表达能力
- **参数共享**：大幅减少参数量（相比全连接层）

In [1]:
# 卷积操作示例
import torch
import torch.nn as nn

# 创建一个简单的卷积层
conv_layer = nn.Conv2d(
    in_channels=1,      # 输入通道数（灰度图像）
    out_channels=1,     # 输出通道数
    kernel_size=3,      # 3×3卷积核
    stride=1,           # 步长为1
    padding=0,          # 不填充
    bias=False          # 不使用偏置
)

# 手动设置卷积核参数（边缘检测卷积核）
with torch.no_grad():
    conv_layer.weight = nn.Parameter(torch.tensor([
        [[[-1., -1., -1.],
          [-1.,  8., -1.],
          [-1., -1., -1.]]]
    ]))

# 创建一个简单的5×5输入图像
input_image = torch.tensor([
    [[[0., 0., 0., 0., 0.],
      [0., 1., 1., 1., 0.],
      [0., 1., 1., 1., 0.],
      [0., 1., 1., 1., 0.],
      [0., 0., 0., 0., 0.]]]
])

print("输入图像形状:", input_image.shape)  # [1, 1, 5, 5]
print("输入图像:\n", input_image.squeeze())

# 执行卷积操作
output = conv_layer(input_image)

print("\n卷积核（边缘检测）:\n", conv_layer.weight.squeeze())
print("\n输出特征图形状:", output.shape)  # [1, 1, 3, 3]
print("输出特征图:\n", output.squeeze())

输入图像形状: torch.Size([1, 1, 5, 5])
输入图像:
 tensor([[0., 0., 0., 0., 0.],
        [0., 1., 1., 1., 0.],
        [0., 1., 1., 1., 0.],
        [0., 1., 1., 1., 0.],
        [0., 0., 0., 0., 0.]])

卷积核（边缘检测）:
 tensor([[-1., -1., -1.],
        [-1.,  8., -1.],
        [-1., -1., -1.]], grad_fn=<SqueezeBackward0>)

输出特征图形状: torch.Size([1, 1, 3, 3])
输出特征图:
 tensor([[5., 3., 5.],
        [3., 0., 3.],
        [5., 3., 5.]], grad_fn=<SqueezeBackward0>)


## 6. PyTorch中常用的卷积层类型

### nn.Conv1d - 一维卷积
- 用于**序列数据**（时间序列、文本、音频等）
- 输入形状：`[batch_size, in_channels, length]`
- 应用：文本分类、语音识别

### nn.Conv2d - 二维卷积
- 用于**图像数据**
- 输入形状：`[batch_size, in_channels, height, width]`
- 应用：图像分类、目标检测、图像分割

### nn.Conv3d - 三维卷积
- 用于**视频或3D数据**
- 输入形状：`[batch_size, in_channels, depth, height, width]`
- 应用：视频分析、医学影像（CT、MRI）

### 转置卷积（nn.ConvTranspose2d）
- 也称为**反卷积**或**上采样卷积**
- 用于**增加特征图尺寸**
- 应用：图像生成、语义分割的解码器

### 深度可分离卷积
- **Depthwise卷积** + **Pointwise卷积**
- 大幅减少参数量和计算量
- 应用：MobileNet、EfficientNet等轻量级网络

---

### 参数量对比示例

**标准卷积** vs **全连接层**：

假设输入：32×32图像，3通道，输出64通道

```python
# 全连接层
fc = nn.Linear(32*32*3, 64*30*30)
# 参数量: 3072 × 57600 = 176,947,200

# 卷积层
conv = nn.Conv2d(3, 64, kernel_size=3, padding=1)
# 参数量: 3 × 64 × 3 × 3 = 1,728
```

**卷积层参数量是全连接层的 0.001%！** 这就是参数共享的威力。

In [10]:
# 演示不同卷积参数的效果
import torch
import torch.nn as nn

# 创建一个示例输入：1张3通道的32×32图像
input_tensor = torch.randn(1, 3, 32, 32)
print(f"输入形状: {input_tensor.shape}\n")

# 1. 标准卷积（保持尺寸）
conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1)
out1 = conv1(input_tensor)
print(f"标准卷积 (k=3, s=1, p=1): {out1.shape}")

# 2. 步长为2（尺寸减半）
conv2 = nn.Conv2d(3, 64, kernel_size=3, stride=2, padding=1)
out2 = conv2(input_tensor)
print(f"步长为2 (k=3, s=2, p=1): {out2.shape}")

# 3. 大卷积核（5×5）
conv3 = nn.Conv2d(3, 64, kernel_size=5, stride=1, padding=2)
out3 = conv3(input_tensor)
print(f"大卷积核 (k=5, s=1, p=2): {out3.shape}")

# 4. 无填充（尺寸缩小）
conv4 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=0)
out4 = conv4(input_tensor)
print(f"无填充 (k=3, s=1, p=0): {out4.shape}")

# 5. 1×1卷积（改变通道数）
conv5 = nn.Conv2d(3, 64, kernel_size=1, stride=1, padding=0)
out5 = conv5(input_tensor)
print(f"1×1卷积 (k=1, s=1, p=0): {out5.shape}")

print("\n💡 观察：")
print("- padding=1 配合 kernel=3, stride=1 可以保持尺寸")
print("- stride=2 可以将尺寸减半（下采样）")
print("- 1×1卷积常用于改变通道数而不改变空间尺寸")

输入形状: torch.Size([1, 3, 32, 32])

标准卷积 (k=3, s=1, p=1): torch.Size([1, 64, 32, 32])
步长为2 (k=3, s=2, p=1): torch.Size([1, 64, 16, 16])
大卷积核 (k=5, s=1, p=2): torch.Size([1, 64, 32, 32])
无填充 (k=3, s=1, p=0): torch.Size([1, 64, 30, 30])
1×1卷积 (k=1, s=1, p=0): torch.Size([1, 64, 32, 32])

💡 观察：
- padding=1 配合 kernel=3, stride=1 可以保持尺寸
- stride=2 可以将尺寸减半（下采样）
- 1×1卷积常用于改变通道数而不改变空间尺寸


## 7. 卷积神经网络常见模块

### 基本卷积块（Conv Block）
标准的卷积块通常包含：
1. **卷积层**（Conv2d）
2. **批归一化**（BatchNorm2d） - 加速训练，稳定梯度
3. **激活函数**（ReLU/LeakyReLU） - 引入非线性

```python
class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, 3, padding=1)
        self.bn = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
    
    def forward(self, x):
        return self.relu(self.bn(self.conv(x)))
```

### 常见卷积模式

#### 1. 下采样（Downsampling）
```python
# 方式1：stride=2
nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1)

# 方式2：MaxPooling
nn.MaxPool2d(kernel_size=2, stride=2)
```

#### 2. 上采样（Upsampling）
```python
# 方式1：转置卷积
nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)

# 方式2：插值 + 卷积
nn.Upsample(scale_factor=2, mode='bilinear')
```

#### 3. 残差连接（Residual Connection）
```python
class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, 3, padding=1)
        self.conv2 = nn.Conv2d(channels, channels, 3, padding=1)
        self.relu = nn.ReLU()
    
    def forward(self, x):
        residual = x
        out = self.relu(self.conv1(x))
        out = self.conv2(out)
        out += residual  # 跳跃连接
        return self.relu(out)
```

### 经典CNN架构演进
- **LeNet-5** (1998): 最早的CNN，用于手写数字识别
- **AlexNet** (2012): 深度CNN的突破，使用ReLU和Dropout
- **VGGNet** (2014): 使用小卷积核（3×3）堆叠
- **ResNet** (2015): 引入残差连接，解决深度网络退化问题
- **MobileNet** (2017): 深度可分离卷积，轻量化模型
- **EfficientNet** (2019): 平衡深度、宽度和分辨率

In [ ]:
# 完整的卷积神经网络示例（用于CIFAR-10图像分类）
import torch
import torch.nn as nn

class SimpleCNN(nn.Module):
    """简单的卷积神经网络"""
    def __init__(self, num_classes=10):
        super(SimpleCNN, self).__init__()
        
        # 特征提取层
        self.features = nn.Sequential(
            # Block 1: 3 -> 64, 32×32 -> 32×32
            nn.Conv2d(3, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),  # 32×32 -> 16×16
            
            # Block 2: 64 -> 128, 16×16 -> 8×8
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),  # 16×16 -> 8×8
            
            # Block 3: 128 -> 256, 8×8 -> 4×4
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),  # 8×8 -> 4×4
        )
        
        # 分类层
        self.classifier = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(256 * 4 * 4, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )
    
    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)  # 展平
        x = self.classifier(x)
        return x

# 创建模型并测试
model = SimpleCNN(num_classes=10)
print(model)

# 测试前向传播
test_input = torch.randn(4, 3, 32, 32)  # 4张32×32的RGB图像
output = model(test_input)
print(f"\n输入形状: {test_input.shape}")
print(f"输出形状: {output.shape}")  # [4, 10] - 4张图像，每张10个类别的预测

# 计算参数量
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\n总参数量: {total_params:,}")
print(f"可训练参数量: {trainable_params:,}")